# Ariel 2025 — Notebook 1: Feature Extraction

Trích **physics-based features** từ raw detector parquet (AIRS + FGS) và lưu thành CSV.
Đây là bước **nặng nhất** — chạy một lần, lưu output (Save Version) rồi dùng lại ở Notebook 2.

Pipeline: ADC → bad-pixel → dark → flat → CDS → temporal binning → light curves → transit detection → features.

In [ ]:
# === Setup: clone running branch and make ariel_ml importable ===
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Jun1801/ML_IT3190E_Project.git"
CLONE_DIR = Path("/kaggle/working/ML_IT3190E_Project")
if not CLONE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", "running", "--single-branch", REPO_URL, str(CLONE_DIR)],
        check=True,
    )
    print("Cloned branch 'running' →", CLONE_DIR)
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull"], check=True)
    print("Pulled latest →", CLONE_DIR)

for _p in [str(CLONE_DIR / "src"), "/kaggle/input/ariel-ml-src/src", "/kaggle/usr/lib/ariel_ml", "src", "../src"]:
    if Path(_p).exists():
        sys.path.insert(0, _p); print("Using ariel_ml from:", _p); break
else:
    print("WARNING: ariel_ml source not found.")

DATA_ROOT = Path("/kaggle/input/ariel-data-challenge-2025")
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT exists:", DATA_ROOT.exists())

## 1. Cấu hình build features
`LIMIT` = số planet (None = full). `TIME_BINS` = số điểm thời gian sau downsample.
Bắt đầu nhỏ để smoke test, sau đó tăng dần (xem `notebooks/config_run_guide.md`).

In [ ]:
from ariel_ml.config import DatasetConfig, FeatureConfig, PreprocessConfig
from ariel_ml.dataset_builder import ArielDatasetBuilder
from ariel_ml.io import ArielDataRepository
from ariel_ml.pipeline import ArielPreprocessFeaturePipeline

# ---- knobs ----
LIMIT = 100          # smoke=5, debug=50, baseline=200-500, full=None
TIME_BINS = 128      # smoke=20, debug=64, baseline/full=128 (256 nếu đủ RAM)
BUILD_TEST = True    # cũng build features cho test split (cần cho submission)

dataset_config = DatasetConfig(data_root=DATA_ROOT)
preprocess_config = PreprocessConfig(
    target_time_bins=TIME_BINS,
    bin_mode="mean",
    bin_before_spatial_calibration=True,
    apply_cds=True,
    apply_linearity=False,
    smooth_window=5,
    detrend_degree=2,
)
feature_config = FeatureConfig(
    spectral_bin_sizes=(1, 2, 4, 8, 16, 32, 64),
    include_per_wavelength_depths=True,
    include_per_wavelength_noise=False,
)

repository = ArielDataRepository(dataset_config)
pipeline = ArielPreprocessFeaturePipeline(preprocess_config, feature_config)
builder = ArielDatasetBuilder(repository=repository, pipeline=pipeline)
print("Planets available (train):", len(repository.list_planet_ids("train")))


## 2. Build train features → CSV

In [ ]:
train_csv = OUTPUT_DIR / "features_train.csv"
train_result = builder.build_feature_csv(
    split="train",
    output_path=train_csv,
    limit=LIMIT,
    aggregate_observations=True,
    on_error="skip",        # bỏ qua planet lỗi thay vì dừng
)
print("Train features:", train_result.features.shape, "-> saved", train_csv)
print("Failures:", len(train_result.failures))
for f in train_result.failures[:5]:
    print("  ", f)
train_result.features.head()


## 3. (Tùy chọn) Build test features → CSV

In [ ]:
if BUILD_TEST:
    test_csv = OUTPUT_DIR / "features_test.csv"
    test_result = builder.build_feature_csv(
        split="test",
        output_path=test_csv,
        limit=None,
        aggregate_observations=True,
        on_error="skip",
    )
    print("Test features:", test_result.features.shape, "-> saved", test_csv)
else:
    print("Skipped test build (set BUILD_TEST=True to enable).")


## 4. Tiếp theo
Nhấn **Save Version** để lưu `features_train.csv` / `features_test.csv` vào output của notebook.
Sau đó attach output này vào **Notebook 2 — Experiments** (hoặc đặt cùng working dir).